In [ ]:
import json
import os
import numpy as np
import pandas as pd
from chronos import Chronos2Pipeline
from sklearn.metrics import root_mean_squared_error

# ============================================================
# CONFIG
# ============================================================
DAYS_JSON = r"C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\dataset_days.json"
DATA_DIR  = r"C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\DataCleaning\clean"
OUT_DIR   = r"C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\Outputs"
MODEL_DIR = r"C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\Models"

countries = ["Germany", "Ireland", "Portugal"]
days = ["day1", "day2", "day3", "day4", "day5"]

features = [
    "temperature_2m",
    "relative_humidity_2m",
    "wind_speed_10m",
    "precipitation",
    "direct_radiation",
]

# 1 day ahead @ 15-min resolution
PRED_LEN = 96

# Use tail windows so training/prediction stays fast
TAIL_TRAIN_PREDICT  = 10_000
TAIL_TRAIN_FINETUNE = 20_000

# Baseline CV RMSE (zero-shot) for improvement %
baseline_cv_rmse = {
    "Germany": 680.116128,
    "Ireland": 535.503877,
    "Portugal": 287.043478,
}

# Fine-tuning hyperparameters
FT_NUM_STEPS = 1000
FT_LR = 1e-5
FT_BATCH_SIZE = 32
FT_LOGGING_STEPS = 5

# Prediction settings
QUANTILES = [0.1, 0.5, 0.9]
PRED_BATCH_SIZE = 128

# If True, after saving models we reload them and generate prediction CSVs
PREDICT_WITH_SAVED_MODELS = True

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

# ============================================================
# LOAD DAY CUTOFFS
# ============================================================
with open(DAYS_JSON, "r") as f:
    dataset_days = json.load(f)

# ============================================================
# HELPERS
# ============================================================
def get_households(df: pd.DataFrame) -> list[str]:
    homes = [c for c in df.columns if c.startswith("home_")]
    homes = sorted(homes, key=lambda x: int(x.split("_")[1]))
    if not homes:
        raise ValueError("No home_* columns found in dataset.")
    return homes

def make_context_df(train_wide: pd.DataFrame, households: list[str], tail_n: int | None) -> pd.DataFrame:
    df2 = train_wide[households].copy()
    if tail_n is not None:
        df2 = df2.tail(tail_n)

    ctx = (
        df2.reset_index()
           .melt(id_vars=["timestamp"], var_name="item_id", value_name="target")
           .sort_values(["item_id", "timestamp"])
           .reset_index(drop=True)
    )
    return ctx

def to_wide_predictions(pred_long: pd.DataFrame, households: list[str]) -> pd.DataFrame:
    if "predictions" in pred_long.columns:
        val_col = "predictions"
    elif "0.5" in pred_long.columns:
        val_col = "0.5"
    else:
        raise ValueError(f"Can't find point forecast column. Columns are: {list(pred_long.columns)}")

    wide = (
        pred_long.pivot(index="timestamp", columns="item_id", values=val_col)
                 .sort_index()
                 .reindex(households, axis=1)
    )
    wide.columns.name = None
    wide.index.name = "timestamp"
    return wide

def build_finetune_inputs(
    df: pd.DataFrame,
    households: list[str],
    features: list[str],
    cutoff: pd.Timestamp,
    tail_n: int | None = 20_000,
    min_len: int = 512,
):
    df_ft = df.loc[df.index < cutoff].copy()
    df_ft[features] = df_ft[features].ffill().bfill()

    train_inputs = []
    for h in households:
        s = df_ft[h].dropna()
        if len(s) < min_len:
            continue

        X = df_ft.loc[s.index, features].copy()

        if tail_n is not None:
            s = s.tail(tail_n)
            X = X.tail(len(s))

        train_inputs.append({
            "target": s.to_numpy(),
            "past_covariates": {col: X[col].to_numpy() for col in features},
            "future_covariates": {},
        })

    return train_inputs

def predict_crosslearning_one_day(
    pipeline: Chronos2Pipeline,
    df: pd.DataFrame,
    households: list[str],
    country: str,
    day: str,
    dataset_days: dict,
    prediction_length: int,
    tail_train: int,
):
    cutoff = pd.to_datetime(dataset_days[country][day])

    train_wide = df.loc[df.index < cutoff, households].copy()
    if train_wide.empty:
        raise ValueError(f"[{country} {day}] empty training slice, cutoff={cutoff}")

    context_df = make_context_df(train_wide, households, tail_n=tail_train)

    pred_long = pipeline.predict_df(
        df=context_df,
        prediction_length=prediction_length,
        quantile_levels=QUANTILES,
        cross_learning=True,
        batch_size=PRED_BATCH_SIZE,
    )

    pred_wide = to_wide_predictions(pred_long, households)
    return pred_wide

def evaluate_crosslearning_cv(
    pipeline: Chronos2Pipeline,
    df: pd.DataFrame,
    households: list[str],
    country: str,
    days: list[str],
    dataset_days: dict,
    prediction_length: int,
    tail_train: int,
):
    day_rows = []

    for day in days:
        pred_wide = predict_crosslearning_one_day(
            pipeline=pipeline,
            df=df,
            households=households,
            country=country,
            day=day,
            dataset_days=dataset_days,
            prediction_length=prediction_length,
            tail_train=tail_train,
        )

        rmses = []
        for h in households:
            y_pred = pred_wide[h]
            y_true = df.loc[y_pred.index, h]

            mask = (~pd.isna(y_true)) & (~pd.isna(y_pred))
            if mask.sum() == 0:
                continue

            rmses.append(float(root_mean_squared_error(y_true[mask], y_pred[mask])))

        day_rmse = float(np.mean(rmses)) if rmses else np.nan
        day_rows.append({"country": country, "day": day, "rmse": day_rmse})

    rmse_day_df = pd.DataFrame(day_rows)
    cv_mean = float(rmse_day_df["rmse"].mean())
    return rmse_day_df, cv_mean

def save_pipeline_and_meta(pipeline: Chronos2Pipeline, model_dir: str, meta: dict):
    os.makedirs(model_dir, exist_ok=True)

    if hasattr(pipeline, "save_pretrained"):
        pipeline.save_pretrained(model_dir)
    elif hasattr(pipeline, "model") and hasattr(pipeline.model, "save_pretrained"):
        pipeline.model.save_pretrained(model_dir)
    else:
        raise RuntimeError("Cannot save this Chronos pipeline/model with save_pretrained().")

    with open(os.path.join(model_dir, "meta.json"), "w") as f:
        json.dump(meta, f, indent=2)

# ============================================================
# RUN: FINE-TUNE + EVALUATE PER COUNTRY
# ============================================================
finetune_summary = []
saved_model_dirs = {}  # country -> model_dir

for country in countries:
    print("\n==============================")
    print("Country:", country)

    data_path = os.path.join(DATA_DIR, f"dataset_{country.capitalize()}.csv")
    df = pd.read_csv(data_path, index_col="timestamp", parse_dates=True).sort_index()
    households = get_households(df)

    base_pipeline = Chronos2Pipeline.from_pretrained("amazon/chronos-2", device_map="cuda")
    ft_cutoff = pd.to_datetime(dataset_days[country]["day1"])

    train_inputs = build_finetune_inputs(
        df=df,
        households=households,
        features=features,
        cutoff=ft_cutoff,
        tail_n=TAIL_TRAIN_FINETUNE,
    )
    print("Fine-tune series count:", len(train_inputs))

    if len(train_inputs) == 0:
        print("  WARNING: no series available for fine-tuning. Skipping.")
        continue

    # ---- Fine-tune (LoRA -> fallback FULL) ----
    try:
        finetuned_pipeline = base_pipeline.fit(
            inputs=train_inputs,
            finetune_mode="lora",
            prediction_length=PRED_LEN,
            num_steps=FT_NUM_STEPS,
            learning_rate=FT_LR,
            batch_size=FT_BATCH_SIZE,
            logging_steps=FT_LOGGING_STEPS,
        )
        finetune_mode_used = "lora"
    except AttributeError as e:
        print("LoRA failed (likely peft/chronos mismatch):", e)
        print("Falling back to FULL fine-tuning...")
        finetuned_pipeline = base_pipeline.fit(
            inputs=train_inputs,
            finetune_mode="full",
            prediction_length=PRED_LEN,
            num_steps=FT_NUM_STEPS,
            learning_rate=FT_LR,
            batch_size=FT_BATCH_SIZE,
            logging_steps=FT_LOGGING_STEPS,
        )
        finetune_mode_used = "full"

    # ---- Save model ----
    model_dir = os.path.join(MODEL_DIR, f"Chronos2_{finetune_mode_used.upper()}_{country}_steps{FT_NUM_STEPS}")
    meta = {
        "country": country,
        "prediction_length": PRED_LEN,
        "features": features,
        "finetune_mode": finetune_mode_used,
        "num_steps": FT_NUM_STEPS,
        "learning_rate": FT_LR,
        "batch_size": FT_BATCH_SIZE,
        "tail_train_finetune": TAIL_TRAIN_FINETUNE,
        "tail_train_predict": TAIL_TRAIN_PREDICT,
        "finetune_cutoff_day": "day1",
        "finetune_cutoff_timestamp": str(ft_cutoff),
        "quantiles": QUANTILES,
        "cross_learning_eval": True,
    }
    save_pipeline_and_meta(finetuned_pipeline, model_dir, meta)
    saved_model_dirs[country] = model_dir
    print("Saved fine-tuned model to:", model_dir)

    # ---- Evaluate (CV) ----
    rmse_day_df, ft_cv_mean = evaluate_crosslearning_cv(
        pipeline=finetuned_pipeline,
        df=df,
        households=households,
        country=country,
        days=days,
        dataset_days=dataset_days,
        prediction_length=PRED_LEN,
        tail_train=TAIL_TRAIN_PREDICT,
    )

    base = baseline_cv_rmse.get(country, np.nan)
    improvement_pct = (base - ft_cv_mean) / base * 100.0 if np.isfinite(base) else np.nan

    finetune_summary.append({
        "country": country,
        "baseline_cv_rmse": base,
        "finetuned_cv_rmse": ft_cv_mean,
        "improvement_%": improvement_pct,
        "model_dir": model_dir,
        "finetune_mode_used": finetune_mode_used,
    })

    print("\nRMSE by day:")
    print(rmse_day_df)
    print("\nBaseline CV mean RMSE:", base)
    print("Fine-tuned CV mean RMSE:", ft_cv_mean)
    print("Improvement (%):", improvement_pct)

summary_df = pd.DataFrame(finetune_summary)
print("\n==============================")
print("Fine-tuning summary:")
print(summary_df)

summary_path = os.path.join(OUT_DIR, "Chronos2_crosslearn_finetune_summary.csv")
summary_df.to_csv(summary_path, index=False)
print("\nSaved summary to:", summary_path)

# ============================================================
# USE THE SAVED FINETUNED MODELS TO PREDICT (CROSS-LEARNING)
# ============================================================
if PREDICT_WITH_SAVED_MODELS:
    print("\n==============================")
    print("Predicting with SAVED fine-tuned models...")

    for country in countries:
        if country not in saved_model_dirs:
            print(f"  Skipping {country}: no saved model dir in this run.")
            continue

        model_dir = saved_model_dirs[country]
        print(f"\nCountry: {country}")
        print("  Loading model from:", model_dir)

        # Load the finetuned pipeline back from disk
        ft_pipeline_loaded = Chronos2Pipeline.from_pretrained(model_dir, device_map="cuda")

        # Load data
        data_path = os.path.join(DATA_DIR, f"dataset_{country.capitalize()}.csv")
        df = pd.read_csv(data_path, index_col="timestamp", parse_dates=True).sort_index()
        households = get_households(df)

        # Predict for all specified days and save CSV
        for day in days:
            pred_wide = predict_crosslearning_one_day(
                pipeline=ft_pipeline_loaded,
                df=df,
                households=households,
                country=country,
                day=day,
                dataset_days=dataset_days,
                prediction_length=PRED_LEN,
                tail_train=TAIL_TRAIN_PREDICT,
            )

            out_path = os.path.join(OUT_DIR, f"Chronos2_finetune_crosslearn_pred_{country.capitalize()}_{day}.csv")
            pred_wide.to_csv(out_path, index=True, index_label="timestamp")
            print(f"  Saved: {out_path}")


Country: Germany
Fine-tune series count: 28


Could not estimate the number of tokens of the input, floating-point operations will not be computed


Step,Training Loss
5,0.923500
10,1.094400


Saved fine-tuned model to: C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\Models\Chronos2_LORA_Germany_steps10

RMSE by day:
   country   day         rmse
0  Germany  day1  1144.114244
1  Germany  day2   234.968820
2  Germany  day3   658.655856
3  Germany  day4  1111.038765
4  Germany  day5   251.629213

Baseline CV mean RMSE: 680.116128
Fine-tuned CV mean RMSE: 680.0813797274618
Improvement (%): 0.005109167553543904

Country: Ireland
Fine-tune series count: 20


Could not estimate the number of tokens of the input, floating-point operations will not be computed


Step,Training Loss
5,0.842400
10,0.917000


Saved fine-tuned model to: C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\Models\Chronos2_LORA_Ireland_steps10

RMSE by day:
   country   day        rmse
0  Ireland  day1  768.764670
1  Ireland  day2  196.493679
2  Ireland  day3  548.944906
3  Ireland  day4  721.312310
4  Ireland  day5  413.786105

Baseline CV mean RMSE: 535.503877
Fine-tuned CV mean RMSE: 529.8603338863625
Improvement (%): 1.0538753043682483

Country: Portugal
Fine-tune series count: 23


Could not estimate the number of tokens of the input, floating-point operations will not be computed


Step,Training Loss
5,0.896400
10,0.993200


Saved fine-tuned model to: C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\Models\Chronos2_LORA_Portugal_steps10

RMSE by day:
    country   day        rmse
0  Portugal  day1  389.338416
1  Portugal  day2  217.799786
2  Portugal  day3  296.512188
3  Portugal  day4  342.076189
4  Portugal  day5  185.879374

Baseline CV mean RMSE: 287.043478
Fine-tuned CV mean RMSE: 286.3211904247694
Improvement (%): 0.25163002492277825

Fine-tuning summary:
    country  baseline_cv_rmse  finetuned_cv_rmse  improvement_%  \
0   Germany        680.116128         680.081380       0.005109   
1   Ireland        535.503877         529.860334       1.053875   
2  Portugal        287.043478         286.321190       0.251630   

                                           model_dir finetune_mode_used  
0  C:\Users\CR58XM\Documents\GitHub\AAU_learning_...               lora  
1  C:\Users\CR58XM\Documents\GitHub\AAU_learning_...               lora  
2  C:\Users\CR58XM\Documents\GitHub\AA